# Pipeline Two-Phase — Visualização e Análise

**Seções:** Setup | Stage 0 | Stage 1 | Stage 2 | Pipeline Completo | Erros TP/FP/FN | Tiny Objects | Comparativo | Gargalo

---
## 1. Setup

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from PIL import Image

PROJECT  = Path('.')
RAW_IMG  = PROJECT / 'data' / 'raw' / 'images'
RAW_ANN  = PROJECT / 'data' / 'raw' / 'annotations'
CROPS    = PROJECT / 'data' / 'interim' / 'two_phase' / 'crops' / 'test' / 'hold'
PREDS    = PROJECT / 'runs' / 'two_phase' / 'predictions'
CC_DIR   = PROJECT / 'runs' / 'two_phase' / 'carry_classifier'
WD_DIR   = PROJECT / 'runs' / 'two_phase' / 'weapon_crop_detector'
# yolo_crops: images/<split>  e  labels/<split>
YOLO_ROOT = PROJECT / 'data' / 'interim' / 'two_phase' / 'yolo_crops'

df_pred    = pd.read_csv(PREDS / 'test_predictions.csv')
df_summary = pd.read_csv(PREDS / 'test_image_summary.csv')
df_persons = pd.read_csv(PREDS / 'test_person_candidates.csv')
df_cc      = pd.read_csv(CC_DIR / 'metrics.csv')
df_wd      = pd.read_csv(WD_DIR / 'results.csv')

print(f'Predições: {len(df_pred)} weapon detections')
print(f'Imagens:   {len(df_summary)} | {df_summary["gt_weapon_boxes"].gt(0).sum()} com GT weapons')
print(f'Pessoas:   {len(df_persons)} person crops')

In [ ]:
def parse_gt(stem):
    p = RAW_ANN / f'{stem}.xml'
    if not p.exists(): return []
    root = ET.parse(p).getroot()
    return [(float(o.find('bndbox/xmin').text), float(o.find('bndbox/ymin').text),
             float(o.find('bndbox/xmax').text), float(o.find('bndbox/ymax').text))
            for o in root.findall('object')]

def load_image(stem):
    return Image.open(RAW_IMG / f'{stem}.jpg').convert('RGB')

def draw_box(ax, box, color, label=None, lw=2, alpha=1.0, fontsize=8):
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=lw,edgecolor=color,facecolor='none',alpha=alpha))
    if label:
        ax.text(x1,y1-4,label,color=color,fontsize=fontsize,fontweight='bold',
                bbox=dict(facecolor='black',alpha=0.5,pad=1,edgecolor='none'))

def iou(a,b):
    xa=max(a[0],b[0]);ya=max(a[1],b[1]);xb=min(a[2],b[2]);yb=min(a[3],b[3])
    inter=max(0,xb-xa)*max(0,yb-ya)
    ua=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/ua if ua>0 else 0

plt.rcParams.update({'figure.dpi':100,'font.family':'DejaVu Sans'})
print('Utilities OK.')

---
## 2. Stage 0 — Detecção de Pessoas

**yolo11n** pré-treinado detecta todas as pessoas. Cada bbox → crop para Stage 1.

In [ ]:
stems = df_summary[(df_summary['gt_weapon_boxes']>0)&(df_summary['persons_detected']>=3)]\
        .sort_values('persons_detected',ascending=False)['image_stem'].iloc[:3].tolist()

fig,axes = plt.subplots(1,3,figsize=(18,6))
fig.suptitle('Stage 0 — Detecção de Pessoas (yolo11n)',fontsize=14,fontweight='bold')
for ax,stem in zip(axes,stems):
    ax.imshow(load_image(stem)); ax.axis('off')
    for _,p in df_persons[df_persons['image_stem']==stem].iterrows():
        draw_box(ax,(p.person_xmin,p.person_ymin,p.person_xmax,p.person_ymax),'#00BFFF',
                 label=f'{p.person_confidence:.2f}',fontsize=7)
    for g in parse_gt(stem): draw_box(ax,g,'#FF4444',label='GT',fontsize=7)
    n=len(df_persons[df_persons['image_stem']==stem])
    ax.set_title(f'frame_{stem.split("frame_")[1]}  |  {n} pessoas',fontsize=9)
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([0],[0],color='#00BFFF',lw=2,label='Pessoa detectada'),
                    Line2D([0],[0],color='#FF4444',lw=2,label='GT Weapon')],
           loc='lower center',ncol=2,fontsize=9)
plt.tight_layout(rect=[0,0.06,1,1]); plt.show()
total_gt=df_summary['gt_weapon_boxes'].sum()
miss0=df_summary['stage0_expanded_crop_miss'].sum()
print(f'Cobertura Stage 0: {100*(1-miss0/total_gt):.1f}%')

---
## 3. Stage 1 — Carry Classifier

**MobileNetV3-Small** (ImageNet). Zoom na parte inferior (55%) do crop.

In [ ]:
crop_files = sorted(CROPS.glob('*.jpg'))[:18]
fig,axes = plt.subplots(3,6,figsize=(18,9))
fig.suptitle("Stage 1 — Crops 'hold' | zoom_lower_fraction=0.55",fontsize=12,fontweight='bold')
for idx,ax in enumerate(axes.flat):
    if idx>=len(crop_files): ax.axis('off'); continue
    img=Image.open(crop_files[idx]).convert('RGB'); w,h=img.size; top=int(round(h*0.45))
    ax.imshow(img.crop((0,top,w,h))); ax.axis('off')
    parts=crop_files[idx].stem.split('_')
    ax.set_title('_'.join(parts[-3:]),fontsize=6)
fig.text(0.5,0.01,'Região inferior (55%) — zona cintura/mãos onde a arma aparece.',
         ha='center',fontsize=9,style='italic')
plt.tight_layout(rect=[0,0.04,1,1]); plt.show()

In [ ]:
sample=list(CROPS.glob('*.jpg'))[:4]
fig,axes=plt.subplots(2,4,figsize=(14,6))
fig.suptitle('Crop Completo vs Zoom 55% Inferior',fontsize=12,fontweight='bold')
for col,cp in enumerate(sample):
    img=Image.open(cp).convert('RGB'); w,h=img.size; top=int(round(h*0.45))
    axes[0,col].imshow(img); axes[0,col].axis('off'); axes[0,col].set_title('Completo',fontsize=8)
    axes[0,col].add_patch(patches.Rectangle((0,top),w,h-top,lw=2,edgecolor='yellow',facecolor='yellow',alpha=0.15))
    axes[0,col].add_patch(patches.Rectangle((0,top),w,h-top,lw=2,edgecolor='yellow',facecolor='none'))
    axes[1,col].imshow(img.crop((0,top,w,h))); axes[1,col].axis('off')
    axes[1,col].set_title('Zoom 55%',fontsize=8,color='goldenrod')
plt.tight_layout(); plt.show()

In [ ]:
ev=df_cc[df_cc['kind']=='epoch_summary'].copy(); ev['epoch']=ev['epoch'].astype(int)
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Carry Classifier — Curvas de Treinamento',fontsize=12,fontweight='bold')
axes[0].plot(ev['epoch'],ev['loss'],'b-o',ms=3,label='Val Loss')
axes[0].plot(ev['epoch'],ev['train_loss'],'r--',ms=3,label='Train Loss',alpha=0.7)
axes[0].axvline(5,color='gray',ls=':',alpha=0.7,label='Unfreeze')
axes[0].set_title('Loss'); axes[0].legend(fontsize=8); axes[0].grid(True,alpha=0.3)
axes[1].plot(ev['epoch'],ev['stage1_gate_recall'],'g-o',ms=3,label='Recall')
axes[1].plot(ev['epoch'],ev['stage1_gate_precision'],'m-o',ms=3,label='Precision')
axes[1].plot(ev['epoch'],ev['stage1_gate_f1'],'b-o',ms=3,label='F1')
axes[1].axhline(0.80,color='red',ls='--',alpha=0.5,label='floor 0.80')
axes[1].axvline(5,color='gray',ls=':',alpha=0.7)
axes[1].set_title('Val — Gate Metrics'); axes[1].legend(fontsize=8); axes[1].set_ylim(0,1); axes[1].grid(True,alpha=0.3)
axes[2].plot(ev['epoch'],ev['stage1_gate_threshold'],color='orange',marker='o',ms=3)
axes[2].set_title('Gate Threshold (val)'); axes[2].set_ylim(0,1); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
best=ev.loc[ev['stage1_gate_f1'].idxmax()]
print(f'Melhor época {int(best["epoch"])} | thr={best["stage1_gate_threshold"]:.2f}')
print(f'Val  P={best["stage1_gate_precision"]:.3f} R={best["stage1_gate_recall"]:.3f} F1={best["stage1_gate_f1"]:.3f}')
print('Test P=0.636 R=0.372 F1=0.469  <- gap generalização (Cam5 != Cam1+Cam7)')

---
## 4. Stage 2 — Weapon Crop Detector

**yolo26n** 120 épocas em crops Cam1+Cam7. mAP50: 0.168 → 0.786 (+368%).

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Stage 2 — Weapon Crop Detector (yolo26n, 120 épocas)',fontsize=12,fontweight='bold')
axes[0].plot(df_wd['epoch'],df_wd['metrics/precision(B)'],'b-',lw=1.5,label='Precision')
axes[0].plot(df_wd['epoch'],df_wd['metrics/recall(B)'],'g-',lw=1.5,label='Recall')
axes[0].set_title('Precision & Recall (val)'); axes[0].legend(fontsize=9); axes[0].set_ylim(0,1); axes[0].grid(True,alpha=0.3)
best_ep=df_wd.loc[df_wd['metrics/mAP50(B)'].idxmax(),'epoch']
best_map=df_wd['metrics/mAP50(B)'].max()
axes[1].plot(df_wd['epoch'],df_wd['metrics/mAP50(B)'],'orange',lw=1.5,label='mAP50')
axes[1].plot(df_wd['epoch'],df_wd['metrics/mAP50-95(B)'],'red',lw=1.5,label='mAP50-95',alpha=0.8)
axes[1].axvline(best_ep,color='gray',ls='--',alpha=0.7,label=f'best={int(best_ep)}')
axes[1].set_title('mAP (val)'); axes[1].legend(fontsize=9); axes[1].set_ylim(0,1); axes[1].grid(True,alpha=0.3)
axes[2].plot(df_wd['epoch'],df_wd['train/box_loss'],'b-',lw=1.5,label='train')
axes[2].plot(df_wd['epoch'],df_wd['val/box_loss'],'b--',lw=1.5,label='val',alpha=0.7)
axes[2].set_title('Box Loss'); axes[2].legend(fontsize=9); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
print(f'mAP50 max: {best_map:.3f} @ época {int(best_ep)}')
print('AVISO: mAP50=0.786 é no val (Cam1+Cam7). No test (Cam5) cai por domain shift.')

In [ ]:
# Crops val com GT weapon  —  estrutura correta: images/val  e  labels/val
val_img = YOLO_ROOT / 'images' / 'val'
val_lbl = YOLO_ROOT / 'labels' / 'val'
annotated=[c for c in sorted(val_img.glob('*.jpg'))
           if (val_lbl/(c.stem+'.txt')).exists() and (val_lbl/(c.stem+'.txt')).stat().st_size>0]
print(f'Crops com arma no val: {len(annotated)}')
n=min(8,len(annotated))
fig,axes=plt.subplots(2,4,figsize=(16,8))
fig.suptitle('Stage 2 — Crops com GT Weapon (val set)',fontsize=12,fontweight='bold')
for idx,ax in enumerate(axes.flat):
    if idx>=n: ax.axis('off'); continue
    cp=annotated[idx]; img=Image.open(cp).convert('RGB'); w,h=img.size
    ax.imshow(img); ax.axis('off')
    with open(val_lbl/(cp.stem+'.txt')) as f:
        for line in f:
            parts=line.strip().split()
            if len(parts)==5:
                _,cx,cy,nw,nh=map(float,parts)
                x1=(cx-nw/2)*w;y1=(cy-nh/2)*h;x2=(cx+nw/2)*w;y2=(cy+nh/2)*h
                draw_box(ax,(x1,y1,x2,y2),'#FF4444',label='GT',fontsize=7)
                ax.set_title(f'{x2-x1:.0f}x{y2-y1:.0f}px',fontsize=8,color='#FF4444')
plt.tight_layout(); plt.show()

---
## 5. Pipeline Completo — Exemplo por Imagem

In [ ]:
def visualize_pipeline(stem,max_crops=5):
    img=load_image(stem); gt=parse_gt(stem)
    persons=df_persons[df_persons['image_stem']==stem]
    preds=df_pred[df_pred['image_stem']==stem]
    nc=min(len(persons),max_crops)
    fig=plt.figure(figsize=(20,5))
    gs=gridspec.GridSpec(1,2+nc,figure=fig,wspace=0.04)
    ax0=fig.add_subplot(gs[0,0]); ax0.imshow(img); ax0.axis('off')
    ax0.set_title(f'Stage 0: {len(persons)} pessoas',fontsize=9,fontweight='bold')
    for _,p in persons.iterrows():
        draw_box(ax0,(p.person_xmin,p.person_ymin,p.person_xmax,p.person_ymax),'#00BFFF',lw=1.5)
    for g in gt: draw_box(ax0,g,'#FF4444',lw=2)
    for ci,(_,p) in enumerate(persons.head(nc).iterrows()):
        ax=fig.add_subplot(gs[0,1+ci])
        crop=img.crop((int(p.crop_xmin),int(p.crop_ymin),int(p.crop_xmax),int(p.crop_ymax))).resize((128,256))
        ax.imshow(crop); ax.axis('off')
        wp=preds[preds['person_index']==p.person_index]
        ax.set_title(f'🔫{wp["weapon_confidence"].max():.2f}' if len(wp) else 'miss',
                     fontsize=8,color='green' if len(wp) else 'gray',fontweight='bold')
    axf=fig.add_subplot(gs[0,-1]); axf.imshow(img); axf.axis('off')
    axf.set_title(f'Final: {len(preds)} detecções',fontsize=9,fontweight='bold')
    for _,pw in preds.iterrows():
        draw_box(axf,(pw.xmin,pw.ymin,pw.xmax,pw.ymax),'#00FF88',label=f'{pw.weapon_confidence:.2f}',fontsize=7)
    for g in gt: draw_box(axf,g,'#FF4444',lw=2,alpha=0.7)
    fig.suptitle(f'Pipeline — frame {stem.split("frame_")[-1]}  |  GT: {len(gt)} weapons',fontsize=11,fontweight='bold')
    plt.show()

tp_stems=df_summary[(df_summary['gt_weapon_boxes']>0)&(df_summary['final_weapon_detections']>0)]\
         .sort_values('final_weapon_detections',ascending=False)['image_stem'].head(3).tolist()
for s in tp_stems: visualize_pipeline(s)

---
## 6. Análise de Erros — TP / FP / FN

In [ ]:
def classify_preds(stem,thr=0.50):
    gts=parse_gt(stem); pi=df_pred[df_pred['image_stem']==stem]
    if not len(pi): return [],[],gts
    mg,md,tp=set(),set(),[]
    for di,(_,d) in enumerate(pi.iterrows()):
        db=(d.xmin,d.ymin,d.xmax,d.ymax); bi,bg=-1,0
        for gi,g in enumerate(gts):
            if gi in mg: continue
            v=iou(db,g)
            if v>bg: bg,bi=v,gi
        if bg>=thr: mg.add(bi);md.add(di);tp.append((db,gts[bi],bg))
    fp=[(d.xmin,d.ymin,d.xmax,d.ymax) for di,(_,d) in enumerate(pi.iterrows()) if di not in md]
    fn=[gts[gi] for gi in range(len(gts)) if gi not in mg]
    return tp,fp,fn

def show_grid(title,stems,etype,n=3):
    fig,axes=plt.subplots(1,n,figsize=(6*n,5))
    fig.suptitle(title,fontsize=12,fontweight='bold')
    if n==1: axes=[axes]
    for ax,stem in zip(axes,stems[:n]):
        ax.imshow(load_image(stem)); ax.axis('off')
        tp,fp,fn=classify_preds(stem)
        if etype in ('tp','all'):
            for det,g,sc in tp: draw_box(ax,det,'#00FF88',label=f'TP{sc:.2f}',fontsize=7); draw_box(ax,g,'#FF4444',lw=1,alpha=0.5)
        if etype in ('fp','all'):
            for f in fp: draw_box(ax,f,'#FF8800',label='FP',fontsize=7)
        if etype in ('fn','all'):
            for f in fn: draw_box(ax,f,'#FF4444',label='FN',fontsize=7)
        ax.set_title(f'TP={len(tp)} FP={len(fp)} FN={len(fn)}',fontsize=9)
    plt.tight_layout(); plt.show()

show_grid('True Positives',tp_stems,'tp')
fp_stems=sorted([(s,len(classify_preds(s)[1])) for s in df_summary[df_summary['final_weapon_detections']>0]['image_stem']],key=lambda x:-x[1])
show_grid('False Positives',[s for s,_ in fp_stems],'fp')
fn_stems=df_summary[(df_summary['gt_weapon_boxes']>0)&(df_summary['final_weapon_detections']==0)&(df_summary['persons_passed_stage2']>0)]['image_stem'].head(3).tolist()
show_grid('False Negatives — Stage 2 miss',fn_stems,'fn')

---
## 7. Por que Stage 2 perde tantas armas?

87% das armas nos crops são tiny objects (<32×32 px em 224×224). YOLO: -38% mAP nessa faixa.

In [ ]:
# Estrutura correta: labels/<split>/*.txt
weapon_sizes=[]
for split in ['train','val','test']:
    lbl_dir = YOLO_ROOT / 'labels' / split
    if not lbl_dir.exists(): continue
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                parts=line.strip().split()
                if len(parts)==5:
                    _,cx,cy,nw,nh=map(float,parts)
                    w_px=nw*224; h_px=nh*224
                    weapon_sizes.append({'split':split,'w_px':w_px,'h_px':h_px,'tiny':w_px<32 and h_px<32})

df_sz=pd.DataFrame(weapon_sizes)
print(f'Total weapons: {len(df_sz)}')
if len(df_sz):
    print(f'Tiny (<32x32): {df_sz["tiny"].sum()} ({100*df_sz["tiny"].mean():.1f}%)')
    print(f'Media: {df_sz["w_px"].mean():.1f} x {df_sz["h_px"].mean():.1f} px')
else:
    print('ERRO: nenhuma anotacao encontrada. Verifique YOLO_ROOT:', YOLO_ROOT.resolve())

In [ ]:
if len(df_sz)==0:
    print('Sem dados — pule esta célula.')
else:
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    fig.suptitle('Tamanho das Armas nos Crops (224x224 px)',fontsize=12,fontweight='bold')
    axes[0].hist(df_sz['w_px'],bins=30,color='steelblue',edgecolor='white',alpha=0.8)
    axes[0].axvline(32,color='red',ls='--',lw=2,label='32px (tiny)')
    axes[0].set_title('Largura'); axes[0].legend(fontsize=9); axes[0].grid(True,alpha=0.3)
    cm={'train':'steelblue','val':'orange','test':'red'}
    for sp,grp in df_sz.groupby('split'):
        axes[1].scatter(grp['w_px'],grp['h_px'],alpha=0.3,s=10,color=cm.get(sp,'gray'),label=sp)
    axes[1].axvline(32,color='red',ls='--',lw=1.5,alpha=0.7); axes[1].axhline(32,color='red',ls='--',lw=1.5,alpha=0.7)
    axes[1].set_title('w x h'); axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)
    tn=df_sz['tiny'].sum(); nn=len(df_sz)-tn
    axes[2].pie([tn,nn],labels=[f'Tiny\n{tn} ({100*tn/len(df_sz):.0f}%)',f'Normal\n{nn} ({100*nn/len(df_sz):.0f}%)'],
                colors=['#FF6B6B','#4ECDC4'],startangle=90)
    axes[2].set_title('Tiny Objects')
    plt.tight_layout(); plt.show()

---
## 8. Comparativo Quantitativo

In [ ]:
cfgs={'Single-Stage\nyolo26n_img9604':dict(tp=266,fp=349,fn=1247,p=0.433,r=0.176,f1=0.250),
      'Two-Phase\n2 epocas':dict(tp=5,fp=690,fn=1508,p=0.007,r=0.003,f1=0.005),
      'Two-Phase\nSAHI+1280 shift':dict(tp=5,fp=690,fn=1508,p=0.007,r=0.003,f1=0.005),
      'Two-Phase\n640@640 BEST':dict(tp=327,fp=153,fn=1186,p=0.681,r=0.216,f1=0.328),
      'Two-Phase\n1280@640px':dict(tp=323,fp=170,fn=1190,p=0.655,r=0.213,f1=0.322),
      'Two-Phase\n1280@1280px':dict(tp=323,fp=170,fn=1190,p=0.655,r=0.213,f1=0.322)}
names=list(cfgs.keys()); colors=['#9E9E9E','#FF6B6B','#FF6B6B','#4CAF50','#81C784','#81C784']
x=np.arange(len(names))
fig,axes=plt.subplots(2,2,figsize=(14,9))
fig.suptitle('Comparativo — Todas as Configurações',fontsize=13,fontweight='bold')
for ax,(key,ylabel) in zip(axes.flat,[('f1','F1'),('p','Precision'),('r','Recall'),('fp','False Positives')]):
    vals=[v[key] for v in cfgs.values()]
    bars=ax.bar(x,vals,width=0.6,color=colors,edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(names,rotation=30,ha='right',fontsize=7)
    ax.set_title(ylabel,fontsize=10,fontweight='bold'); ax.grid(axis='y',alpha=0.3)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+max(vals)*0.01,
                f'{val:.3f}' if val<5 else str(int(val)),ha='center',va='bottom',fontsize=7,fontweight='bold')
from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor='#9E9E9E',label='Single-Stage'),Patch(facecolor='#4CAF50',label='Two-Phase BEST'),
                    Patch(facecolor='#81C784',label='Two-Phase outras'),Patch(facecolor='#FF6B6B',label='Configs falha')],
           loc='upper right',fontsize=8,ncol=2)
plt.tight_layout(rect=[0,0,0.98,0.95]); plt.show()
print('F1: 0.250 -> 0.328 (+31%) | Precision: +57% | FP: 349 -> 153 (-56%)')

---
## 9. Gargalo do Pipeline

In [ ]:
total_gt=1513; s0=36; s2=1146; nms=4; det=327
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('Destino das 1513 GT Weapons',fontsize=13,fontweight='bold')
axes[0].pie([det,s2,s0,nms],
    labels=[f'Detectadas\n{det} (21.6%)',f'Stage 2 miss\n{s2} (75.7%)',f'Stage 0 miss\n{s0} (2.4%)',f'NMS\n{nms}'],
    colors=['#4CAF50','#FF5252','#FF9800','#9E9E9E'],explode=[0.05,0.05,0,0],startangle=90,textprops={'fontsize':9})
axes[0].set_title('Distribuição das Perdas')
stages=['Total GT','Após Stage 0','Após Stage 1\n(desativado)','Após Stage 2','Após NMS']
values=[total_gt,total_gt-s0,total_gt-s0,det+nms,det]
bars=axes[1].bar(stages,values,color=['#2196F3','#FF9800','#9E9E9E','#FF5252','#4CAF50'],edgecolor='white',width=0.6)
for bar,val in zip(bars,values):
    axes[1].text(bar.get_x()+bar.get_width()/2,val+10,str(val),ha='center',fontsize=10,fontweight='bold')
axes[1].set_ylabel('GT weapons restantes'); axes[1].set_title('Funil'); axes[1].grid(axis='y',alpha=0.3)
axes[1].tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.show()
print('Stage 0: ok (97.6%) | Stage 1: desativado | Stage 2: GARGALO (75.7% miss)')
print('Causa: domain shift Cam1+Cam7 -> Cam5 + 87% das armas sao tiny objects')

In [ ]:
cam1=sorted(RAW_IMG.glob('Cam1*.jpg'))[:3]
cam7=sorted(RAW_IMG.glob('Cam7*.jpg'))[:3]
cam5=sorted(RAW_IMG.glob('Cam5*.jpg'))[:3]
rows=[]
if cam1: rows.append(('Cam1 (treino)',cam1,'#4CAF50'))
if cam7: rows.append(('Cam7 (treino)',cam7,'#2196F3'))
rows.append(('Cam5 — TEST (domain shift)',cam5,'#FF5252'))
fig,axes=plt.subplots(len(rows),3,figsize=(15,4*len(rows)))
if len(rows)==1: axes=[axes]
fig.suptitle('Domain Shift — Diferença Visual entre Câmeras',fontsize=12,fontweight='bold')
for ri,(cam_name,imgs,color) in enumerate(rows):
    for ci,ip in enumerate(imgs):
        ax=axes[ri][ci]; ax.imshow(Image.open(ip).convert('RGB')); ax.axis('off')
        ax.set_title(ip.stem.split('frame_')[-1],fontsize=7)
        if ci==0: ax.set_ylabel(cam_name,fontsize=9,color=color,fontweight='bold',rotation=90,labelpad=10)
        for spine in ax.spines.values(): spine.set_edgecolor(color);spine.set_linewidth(3);spine.set_visible(True)
plt.tight_layout(); plt.show()
print('Weapon detector treinado Cam1+Cam7, testado Cam5 — domain shift explica Stage 2 miss.')